# JN0g · Building with agents

*On-ramp 7 of 8.*

You're not a programmer. Can you still build — and *check* — analyses like this, by directing an AI agent in plain English? Increasingly, yes. This is where the course is taking you.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, the permit feed, and the project's shared code. The two knobs near the top are all a student changes to run another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## The shift: you describe intent, the agent writes the code

An **AI coding agent** reads your goal in plain language, inspects the project, writes and runs code, and shows you the result — you review the *meaning*, not the syntax. Today's agents include **Claude Code, Codex, Qwen, DeepSeek, and Copilot.**

## The nine-step cycle

On a serious project the loop has nine steps:

1. **You** write the spec (what you want).
2. The agent inspects the repo and proposes a **plan**.
3. You approve or edit the plan.
4. The agent implements a small **diff** (a focused change).
5. The agent runs **tests / typecheck / lint** (automated correctness checks).
6. A *second* agent **reviews** the diff.
7. **You** review semantic correctness (does it mean the right thing?).
8. **CI** (continuous integration — automated gates) guards the merge.
9. The agent writes the release notes / docs.

*Forward-looking:* a small project runs a lighter version of this (you are both reviewer and gate) — but the nine steps are the destination, so we name them now.

## The horizon: a journalist ships a verified finding

Picture a reporter who has never written SQL. She points an agent at a **Datasette** of Berkeley permits and asks, in English, *"how many homes were completed each year?"* The agent drafts the SQL; she reads it, runs it, and — crucially — **checks the answer against a number she already trusts** before publishing. Let's do that last, decisive step ourselves.

In [4]:
import sqlite3
DB = REPO_ROOT/'databases/hcd_apr_mirror_2026-06-17_fresh.db'
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
# the kind of SQL an agent drafts for 'completed homes in 2024' — sum the 11 certificate-of-
# occupancy income columns for that year:
co = ['CO_ACUTELY_LOW_INCOME_DR','CO_ACUTELY_LOW_INCOME_NDR','CO_EXTREMELY_LOW_INCOME_DR',
      'CO_EXTREMELY_INCOME_NDR','CO_VLOW_INCOME_DR','CO_VLOW_INCOME_NDR','CO_LOW_INCOME_DR',
      'CO_LOW_INCOME_NDR','CO_MOD_INCOME_DR','CO_MOD_INCOME_NDR','CO_ABOVE_MOD_INCOME']
expr = '+'.join(f'COALESCE(CAST({c} AS INT),0)' for c in co)
got = con.execute(f'SELECT SUM({expr}) FROM table_a2 WHERE YEAR=2024').fetchone()[0]
print('the agent-style query returns:', got, 'completed homes for 2024')

the agent-style query returns: 708 completed homes for 2024


In [5]:
md(f'''### Verification is the whole point

The query returns **{got}** completed homes for 2024 — and that matches the **708** Berkeley reported to the state for that year. The agent wrote the SQL; the *check against a known number* is what makes the finding trustworthy. An answer you didn't verify is a guess in a lab coat — the loop exists to kill the guess.''')

### Verification is the whole point

The query returns **708** completed homes for 2024 — and that matches the **708** Berkeley reported to the state for that year. The agent wrote the SQL; the *check against a known number* is what makes the finding trustworthy. An answer you didn't verify is a guess in a lab coat — the loop exists to kill the guess.

**Next — JN0h:** an agent is only as good as the rules it's given — which live in an **instruction file**.